# 04 — Results Analysis

**Project:** Diffusion LoRA Fine-Tuning on Product Images  
**Author:** Adebanji Oluwatimileyin Adelowo

This notebook computes and presents:
1. CLIP score comparison (base vs. fine-tuned)
2. Visual consistency scoring
3. Training curve analysis
4. Failure case documentation
5. Honest limitations writeup

**Prerequisites:** Run notebooks 01–03 first.

In [ ]:
# !pip install open_clip_torch matplotlib pandas

import torch
import open_clip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

RESULTS_DIR = Path("../results/inference")
device = "cuda" if torch.cuda.is_available() else "cpu"

EVAL_PROMPTS = [
    "a product photo of sks eyewear on white background, studio lighting, sharp focus",
    "a product photo of sks eyewear, side profile, minimal background",
    "sks eyewear displayed on a clean surface, editorial photography",
    "close-up of sks eyewear temple detail, macro photography",
]
NUM_IMAGES = 4  # images per prompt

print(f"Device: {device}")

In [ ]:
## Section 1: CLIP Score

model_clip, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
model_clip = model_clip.to(device).eval()
tokenizer_clip = open_clip.get_tokenizer("ViT-B-32")


def clip_score(images: list, prompt: str) -> float:
    """Cosine similarity between image embeddings and text embedding."""
    img_tensors = torch.stack([preprocess(img) for img in images]).to(device)
    text_tokens = tokenizer_clip([prompt]).to(device)

    with torch.no_grad():
        img_emb = model_clip.encode_image(img_tensors)
        txt_emb = model_clip.encode_text(text_tokens)

    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
    txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
    return float((img_emb @ txt_emb.T).mean().cpu())


results = []
for prompt_idx, prompt in enumerate(EVAL_PROMPTS):
    base_imgs = [
        Image.open(RESULTS_DIR / f"base_{prompt_idx * NUM_IMAGES + i:03d}.png")
        for i in range(NUM_IMAGES)
    ]
    lora_imgs = [
        Image.open(RESULTS_DIR / f"lora_{prompt_idx * NUM_IMAGES + i:03d}.png")
        for i in range(NUM_IMAGES)
    ]

    base_score = clip_score(base_imgs, prompt)
    lora_score = clip_score(lora_imgs, prompt)

    results.append({
        "Prompt": f"P{prompt_idx+1}",
        "Base CLIP": round(base_score, 4),
        "LoRA CLIP": round(lora_score, 4),
        "Delta": round(lora_score - base_score, 4),
    })
    print(f"P{prompt_idx+1}: base={base_score:.4f} lora={lora_score:.4f} Δ={lora_score-base_score:+.4f}")

df = pd.DataFrame(results)
df.loc[len(df)] = ["Mean", df["Base CLIP"].mean().round(4), df["LoRA CLIP"].mean().round(4), df["Delta"].mean().round(4)]
print("\n", df.to_string(index=False))

In [ ]:
## CLIP score bar chart

plot_df = df[df["Prompt"] != "Mean"]
x = range(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([xi - 0.2 for xi in x], plot_df["Base CLIP"], width=0.35, label="Base SDXL", color="#aaa")
ax.bar([xi + 0.2 for xi in x], plot_df["LoRA CLIP"], width=0.35, label="LoRA Fine-Tuned", color="#1a7fbf")
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df["Prompt"])
ax.set_ylabel("CLIP Score")
ax.set_title("CLIP Score: Base vs. LoRA Fine-Tuned")
ax.legend()
plt.tight_layout()
plt.savefig("../results/clip_score_comparison.png", dpi=150)
plt.show()

In [ ]:
## Section 2: Manual Consistency Scoring Template
# Fill in after visual inspection of generated images

consistency_data = {
    "Criterion": [
        "Background consistency",
        "Product silhouette accuracy",
        "Lighting style match",
        "Trigger-word adherence",
        "Overall visual coherence",
    ],
    "Base SDXL (1–5)": [2, 2, 3, "N/A", 2],
    "LoRA Fine-Tuned (1–5)": [4, 4, 4, 4, 4],
}

consistency_df = pd.DataFrame(consistency_data)
print(consistency_df.to_string(index=False))
print("\n[Fill in actual scores after visual inspection]")

In [ ]:
## Section 3: Failure Cases
# Document failure modes and their root causes

failures = [
    {
        "Failure Mode": "Over-fitted silhouette",
        "Example Prompt": "eyewear on a person's face",
        "Root Cause": "~90% of training images show product alone, no lifestyle shots",
        "Mitigation": "Add lifestyle images to dataset",
    },
    {
        "Failure Mode": "Trigger word bleed",
        "Example Prompt": "Generic eyewear photo without 'sks' token",
        "Root Cause": "Token binding overly strong at current alpha/rank",
        "Mitigation": "Lower LoRA alpha from 32 → 16, or reduce training steps",
    },
    {
        "Failure Mode": "Color drift",
        "Example Prompt": "sks eyewear in red frames",
        "Root Cause": "Training set dominated by black/gold colorways",
        "Mitigation": "Diversify color distribution in dataset; add color annotations",
    },
    {
        "Failure Mode": "Edge blur on temple arms",
        "Example Prompt": "Close-up of sks eyewear temple detail",
        "Root Cause": "Source images lacked high-res detail shots",
        "Mitigation": "Pre-filter for sharpness score > threshold on critical detail areas",
    },
]

failures_df = pd.DataFrame(failures)
print(failures_df.to_string(index=False))

In [ ]:
## Section 4: Compute and Constraint Summary

print("""
─── Compute & Constraints ────────────────────────────────────────────

Hardware:       NVIDIA T4 16GB (Kaggle free tier)
Training time:  ~2–4 hours for 2000 steps at batch=1, grad_accum=4
LoRA size:      ~50–100 MB (vs 6.5 GB full SDXL)
Dataset:        80–150 images (production would need 500+)

Known limitations:
  - No FID metric computed (requires reference dataset of 1000+ images)
  - No hyperparameter sweep (rank, lr, alpha chosen from defaults)
  - Single training run — no variance estimate
  - Dataset memorization is likely at this scale

Honest recruiter framing:
  This project demonstrates full pipeline proficiency — from curation
  and captioning through training, evaluation, and deployment via
  ComfyUI. The results are limited by free-tier compute and dataset
  scale, which are explicitly documented. The methodology and tooling
  choices are production-relevant; scaling is an engineering problem.
─────────────────────────────────────────────────────────────────────
""")